In [3]:
!pip uninstall -y google-adk
!pip install -q --upgrade \
  opentelemetry-api==1.44.0 \
  opentelemetry-sdk==1.44.0 \
  opentelemetry-exporter-otlp-proto-grpc==1.44.0

Found existing installation: google-adk 2.7.1
Uninstalling google-adk-2.7.1:
  Successfully uninstalled google-adk-2.7.1


In [4]:
import langchain
import chromadb
import pydantic

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph

print("✅ LangChain:", langchain.__version__)
print("✅ ChromaDB:", chromadb.__version__)
print("✅ Pydantic:", pydantic.__version__)
print("✅ LangGraph: OK")
print("✅ Gemini integration: OK")

✅ LangChain: 1.3.15
✅ ChromaDB: 1.5.9
✅ Pydantic: 2.13.4
✅ LangGraph: OK
✅ Gemini integration: OK


In [5]:
# ============================================================
# КРОК 2. Доменні інструменти EnergyAgent
# Pydantic v2 + Field + field_validator + JSON-вихід
# ============================================================

import json
from typing import Literal

from pydantic import BaseModel, Field, field_validator
from langchain_core.tools import tool


# ------------------------------------------------------------
# Тестовий стан енергосистеми.
# У реальному проєкті ці значення могли б надходити
# з Home Assistant, Modbus, MQTT або API інвертора.
# ------------------------------------------------------------

ENERGY_STATE = {
    "inverter_1": {
        "pv_power_w": 4200,
        "load_power_w": 2800,
        "battery_soc": 72,
        "battery_power_w": -900,
        "grid_power_w": -500,
        "power_limit_w": 6000,
    }
}


# ============================================================
# 1. TOOL: отримання поточного стану енергосистеми
# ============================================================

class EnergyStatusInput(BaseModel):
    inverter_id: str = Field(
        ...,
        description="Ідентифікатор інвертора, наприклад inverter_1"
    )

    @field_validator("inverter_id")
    @classmethod
    def validate_inverter_id(cls, value: str) -> str:
        value = value.strip().lower()

        if not value:
            raise ValueError("inverter_id не може бути порожнім")

        if not value.startswith("inverter_"):
            raise ValueError("inverter_id повинен починатися з 'inverter_'")

        return value


@tool(args_schema=EnergyStatusInput)
def get_energy_status(inverter_id: str) -> str:
    """
    Отримує поточний стан енергосистеми:
    генерацію PV, навантаження, SOC батареї,
    потужність батареї, мережі та поточний ліміт інвертора.
    """

    try:
        if inverter_id not in ENERGY_STATE:
            return json.dumps(
                {
                    "status": "error",
                    "error": f"Інвертор {inverter_id} не знайдено"
                },
                ensure_ascii=False
            )

        return json.dumps(
            {
                "status": "success",
                "data": ENERGY_STATE[inverter_id]
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


# ============================================================
# 2. TOOL: розрахунок доступної потужності
# ============================================================

class SafeLoadInput(BaseModel):
    pv_power_w: int = Field(
        ...,
        ge=0,
        le=50000,
        description="Поточна потужність сонячної генерації у ватах"
    )

    current_load_w: int = Field(
        ...,
        ge=0,
        le=50000,
        description="Поточне споживання будинку у ватах"
    )

    reserve_w: int = Field(
        default=500,
        ge=0,
        le=10000,
        description="Резерв потужності, який не можна використовувати"
    )

    @field_validator("reserve_w")
    @classmethod
    def validate_reserve(cls, value: int) -> int:
        if value % 50 != 0:
            raise ValueError("reserve_w повинен бути кратним 50 Вт")
        return value


@tool(args_schema=SafeLoadInput)
def calculate_safe_load(
    pv_power_w: int,
    current_load_w: int,
    reserve_w: int = 500
) -> str:
    """
    Розраховує безпечну додаткову потужність,
    яку можна підключити без використання мережі.
    """

    try:
        available_power = pv_power_w - current_load_w - reserve_w
        available_power = max(0, available_power)

        return json.dumps(
            {
                "status": "success",
                "data": {
                    "available_power_w": available_power,
                    "reserve_w": reserve_w,
                    "can_add_load": available_power > 0
                }
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


# ============================================================
# 3. TOOL: оцінка стану батареї
# ============================================================

class BatteryCheckInput(BaseModel):
    soc: int = Field(
        ...,
        ge=0,
        le=100,
        description="Поточний заряд батареї у відсотках"
    )

    minimum_soc: int = Field(
        default=20,
        ge=5,
        le=90,
        description="Мінімально допустимий SOC батареї"
    )

    @field_validator("minimum_soc")
    @classmethod
    def validate_minimum_soc(cls, value: int) -> int:
        if value < 10:
            raise ValueError(
                "Для цієї системи minimum_soc не рекомендується нижче 10%"
            )
        return value


@tool(args_schema=BatteryCheckInput)
def check_battery_safety(
    soc: int,
    minimum_soc: int = 20
) -> str:
    """
    Перевіряє рівень заряду батареї та визначає,
    чи дозволено подальший розряд.
    """

    try:
        safe = soc > minimum_soc

        return json.dumps(
            {
                "status": "success",
                "data": {
                    "soc": soc,
                    "minimum_soc": minimum_soc,
                    "discharge_allowed": safe,
                    "message": (
                        "Розряд дозволено"
                        if safe
                        else "Розряд необхідно обмежити"
                    )
                }
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


# ============================================================
# 4. TOOL: зміна ліміту інвертора
# РИЗИКОВИЙ TOOL — пізніше підключимо Human-in-the-Loop
# ============================================================

class SetPowerLimitInput(BaseModel):
    inverter_id: str = Field(
        ...,
        description="Ідентифікатор інвертора"
    )

    limit_w: int = Field(
        ...,
        ge=500,
        le=12000,
        description="Новий ліміт активної потужності у ватах"
    )

    reason: str = Field(
        ...,
        min_length=5,
        max_length=200,
        description="Причина зміни ліміту"
    )

    @field_validator("inverter_id")
    @classmethod
    def validate_inverter_id(cls, value: str) -> str:
        value = value.strip().lower()

        if not value.startswith("inverter_"):
            raise ValueError(
                "inverter_id повинен починатися з 'inverter_'"
            )

        return value

    @field_validator("limit_w")
    @classmethod
    def validate_limit(cls, value: int) -> int:
        if value % 100 != 0:
            raise ValueError(
                "Ліміт потужності повинен бути кратним 100 Вт"
            )
        return value


@tool(args_schema=SetPowerLimitInput)
def set_inverter_power_limit(
    inverter_id: str,
    limit_w: int,
    reason: str
) -> str:
    """
    Змінює максимальний ліміт потужності інвертора.

    УВАГА:
    це ризикова операція, тому надалі вона буде
    виконуватися тільки після Human-in-the-Loop підтвердження.
    """

    try:
        if inverter_id not in ENERGY_STATE:
            return json.dumps(
                {
                    "status": "error",
                    "error": f"Інвертор {inverter_id} не знайдено"
                },
                ensure_ascii=False
            )

        old_limit = ENERGY_STATE[inverter_id]["power_limit_w"]
        ENERGY_STATE[inverter_id]["power_limit_w"] = limit_w

        return json.dumps(
            {
                "status": "success",
                "data": {
                    "inverter_id": inverter_id,
                    "old_limit_w": old_limit,
                    "new_limit_w": limit_w,
                    "reason": reason
                }
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


# ------------------------------------------------------------
# Список інструментів, який пізніше передамо ReAct-агенту
# ------------------------------------------------------------

energy_tools = [
    get_energy_status,
    calculate_safe_load,
    check_battery_safety,
    set_inverter_power_limit,
]

print("✅ Створено доменних tools:", len(energy_tools))

for t in energy_tools:
    print(" -", t.name)

✅ Створено доменних tools: 4
 - get_energy_status
 - calculate_safe_load
 - check_battery_safety
 - set_inverter_power_limit


In [6]:
# ============================================================
# КРОК 3. Локальне тестування tools і Pydantic-валідації
# ============================================================

from pydantic import ValidationError

print("=== 1. Тест get_energy_status ===")
print(
    get_energy_status.invoke(
        {"inverter_id": "inverter_1"}
    )
)

print("\n=== 2. Тест calculate_safe_load ===")
print(
    calculate_safe_load.invoke(
        {
            "pv_power_w": 4200,
            "current_load_w": 2800,
            "reserve_w": 500
        }
    )
)

print("\n=== 3. Тест check_battery_safety ===")
print(
    check_battery_safety.invoke(
        {
            "soc": 72,
            "minimum_soc": 20
        }
    )
)

print("\n=== 4. Перевірка Pydantic-валідації ===")

try:
    SafeLoadInput(
        pv_power_w=4200,
        current_load_w=2800,
        reserve_w=525   # навмисно некоректне значення
    )
except ValidationError as e:
    print("✅ ValidationError спрацював:")
    print(e)

print("\n=== 5. Перевірка ризикового tool без зміни стану ===")

test_input = SetPowerLimitInput(
    inverter_id="inverter_1",
    limit_w=5000,
    reason="Тестова перевірка схеми"
)

print("✅ Схема валідна:")
print(test_input.model_dump())

print("\n✅ Локальні тести завершено")

=== 1. Тест get_energy_status ===
{"status": "success", "data": {"pv_power_w": 4200, "load_power_w": 2800, "battery_soc": 72, "battery_power_w": -900, "grid_power_w": -500, "power_limit_w": 6000}}

=== 2. Тест calculate_safe_load ===
{"status": "success", "data": {"available_power_w": 900, "reserve_w": 500, "can_add_load": true}}

=== 3. Тест check_battery_safety ===
{"status": "success", "data": {"soc": 72, "minimum_soc": 20, "discharge_allowed": true, "message": "Розряд дозволено"}}

=== 4. Перевірка Pydantic-валідації ===
✅ ValidationError спрацював:
1 validation error for SafeLoadInput
reserve_w
  Value error, reserve_w повинен бути кратним 50 Вт [type=value_error, input_value=525, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

=== 5. Перевірка ризикового tool без зміни стану ===
✅ Схема валідна:
{'inverter_id': 'inverter_1', 'limit_w': 5000, 'reason': 'Тестова перевірка схеми'}

✅ Локальні тести завершено


In [7]:
# ============================================================
# КРОК 4. Pytest: валідація схем + тести tools
# ============================================================

%%writefile test_energy_agent.py

import json
import pytest
from pydantic import ValidationError

# Імпортуємо об'єкти з поточного notebook через __main__
from __main__ import (
    EnergyStatusInput,
    SafeLoadInput,
    BatteryCheckInput,
    SetPowerLimitInput,
    get_energy_status,
    calculate_safe_load,
    check_battery_safety,
)


# ------------------------------------------------------------
# 1. Валідний inverter_id
# ------------------------------------------------------------
def test_valid_inverter_id():
    model = EnergyStatusInput(inverter_id="inverter_1")
    assert model.inverter_id == "inverter_1"


# ------------------------------------------------------------
# 2. Невалідний inverter_id
# ------------------------------------------------------------
def test_invalid_inverter_id():
    with pytest.raises(ValidationError):
        EnergyStatusInput(inverter_id="device_1")


# ------------------------------------------------------------
# 3. Валідний reserve_w
# ------------------------------------------------------------
def test_valid_reserve():
    model = SafeLoadInput(
        pv_power_w=4200,
        current_load_w=2800,
        reserve_w=500,
    )
    assert model.reserve_w == 500


# ------------------------------------------------------------
# 4. reserve_w не кратний 50
# ------------------------------------------------------------
def test_invalid_reserve():
    with pytest.raises(ValidationError):
        SafeLoadInput(
            pv_power_w=4200,
            current_load_w=2800,
            reserve_w=525,
        )


# ------------------------------------------------------------
# 5. Некоректний SOC > 100
# ------------------------------------------------------------
def test_invalid_soc():
    with pytest.raises(ValidationError):
        BatteryCheckInput(
            soc=120,
            minimum_soc=20,
        )


# ------------------------------------------------------------
# 6. Некоректний minimum_soc < 10
# ------------------------------------------------------------
def test_invalid_minimum_soc():
    with pytest.raises(ValidationError):
        BatteryCheckInput(
            soc=70,
            minimum_soc=5,
        )


# ------------------------------------------------------------
# 7. Некоректний power limit
# ------------------------------------------------------------
def test_invalid_power_limit():
    with pytest.raises(ValidationError):
        SetPowerLimitInput(
            inverter_id="inverter_1",
            limit_w=5555,
            reason="Тест некоректного ліміту",
        )


# ------------------------------------------------------------
# 8. Tool get_energy_status
# ------------------------------------------------------------
def test_get_energy_status_tool():
    result = json.loads(
        get_energy_status.invoke(
            {"inverter_id": "inverter_1"}
        )
    )

    assert result["status"] == "success"
    assert "pv_power_w" in result["data"]


# ------------------------------------------------------------
# 9. Tool calculate_safe_load
# ------------------------------------------------------------
def test_calculate_safe_load_tool():
    result = json.loads(
        calculate_safe_load.invoke(
            {
                "pv_power_w": 4200,
                "current_load_w": 2800,
                "reserve_w": 500,
            }
        )
    )

    assert result["status"] == "success"
    assert result["data"]["available_power_w"] == 900


# ------------------------------------------------------------
# 10. Tool check_battery_safety
# ------------------------------------------------------------
def test_check_battery_safety_tool():
    result = json.loads(
        check_battery_safety.invoke(
            {
                "soc": 72,
                "minimum_soc": 20,
            }
        )
    )

    assert result["status"] == "success"
    assert result["data"]["discharge_allowed"] is True

Writing test_energy_agent.py


In [9]:
%%writefile energy_agent_core.py

import json

from pydantic import BaseModel, Field, field_validator
from langchain_core.tools import tool


ENERGY_STATE = {
    "inverter_1": {
        "pv_power_w": 4200,
        "load_power_w": 2800,
        "battery_soc": 72,
        "battery_power_w": -900,
        "grid_power_w": -500,
        "power_limit_w": 6000,
    }
}


class EnergyStatusInput(BaseModel):
    inverter_id: str = Field(
        ...,
        description="Ідентифікатор інвертора, наприклад inverter_1"
    )

    @field_validator("inverter_id")
    @classmethod
    def validate_inverter_id(cls, value: str) -> str:
        value = value.strip().lower()

        if not value:
            raise ValueError("inverter_id не може бути порожнім")

        if not value.startswith("inverter_"):
            raise ValueError(
                "inverter_id повинен починатися з 'inverter_'"
            )

        return value


@tool(args_schema=EnergyStatusInput)
def get_energy_status(inverter_id: str) -> str:
    """Отримує поточний стан енергосистеми."""

    try:
        if inverter_id not in ENERGY_STATE:
            return json.dumps(
                {
                    "status": "error",
                    "error": f"Інвертор {inverter_id} не знайдено"
                },
                ensure_ascii=False
            )

        return json.dumps(
            {
                "status": "success",
                "data": ENERGY_STATE[inverter_id]
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


class SafeLoadInput(BaseModel):
    pv_power_w: int = Field(..., ge=0, le=50000)
    current_load_w: int = Field(..., ge=0, le=50000)
    reserve_w: int = Field(default=500, ge=0, le=10000)

    @field_validator("reserve_w")
    @classmethod
    def validate_reserve(cls, value: int) -> int:
        if value % 50 != 0:
            raise ValueError(
                "reserve_w повинен бути кратним 50 Вт"
            )
        return value


@tool(args_schema=SafeLoadInput)
def calculate_safe_load(
    pv_power_w: int,
    current_load_w: int,
    reserve_w: int = 500
) -> str:
    """Розраховує безпечну додаткову потужність."""

    try:
        available_power = (
            pv_power_w
            - current_load_w
            - reserve_w
        )

        available_power = max(0, available_power)

        return json.dumps(
            {
                "status": "success",
                "data": {
                    "available_power_w": available_power,
                    "reserve_w": reserve_w,
                    "can_add_load": available_power > 0
                }
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


class BatteryCheckInput(BaseModel):
    soc: int = Field(..., ge=0, le=100)
    minimum_soc: int = Field(default=20, ge=5, le=90)

    @field_validator("minimum_soc")
    @classmethod
    def validate_minimum_soc(cls, value: int) -> int:
        if value < 10:
            raise ValueError(
                "minimum_soc не рекомендується нижче 10%"
            )
        return value


@tool(args_schema=BatteryCheckInput)
def check_battery_safety(
    soc: int,
    minimum_soc: int = 20
) -> str:
    """Перевіряє безпечність розряду батареї."""

    try:
        safe = soc > minimum_soc

        return json.dumps(
            {
                "status": "success",
                "data": {
                    "soc": soc,
                    "minimum_soc": minimum_soc,
                    "discharge_allowed": safe
                }
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


class SetPowerLimitInput(BaseModel):
    inverter_id: str
    limit_w: int = Field(..., ge=500, le=12000)
    reason: str = Field(..., min_length=5, max_length=200)

    @field_validator("inverter_id")
    @classmethod
    def validate_inverter_id(cls, value: str) -> str:
        value = value.strip().lower()

        if not value.startswith("inverter_"):
            raise ValueError(
                "inverter_id повинен починатися з 'inverter_'"
            )

        return value

    @field_validator("limit_w")
    @classmethod
    def validate_limit(cls, value: int) -> int:
        if value % 100 != 0:
            raise ValueError(
                "Ліміт повинен бути кратним 100 Вт"
            )
        return value


@tool(args_schema=SetPowerLimitInput)
def set_inverter_power_limit(
    inverter_id: str,
    limit_w: int,
    reason: str
) -> str:
    """Ризиковий tool зміни ліміту інвертора."""

    try:
        if inverter_id not in ENERGY_STATE:
            return json.dumps(
                {
                    "status": "error",
                    "error": f"Інвертор {inverter_id} не знайдено"
                },
                ensure_ascii=False
            )

        old_limit = ENERGY_STATE[inverter_id]["power_limit_w"]
        ENERGY_STATE[inverter_id]["power_limit_w"] = limit_w

        return json.dumps(
            {
                "status": "success",
                "data": {
                    "old_limit_w": old_limit,
                    "new_limit_w": limit_w,
                    "reason": reason
                }
            },
            ensure_ascii=False
        )

    except Exception as exc:
        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )

Writing energy_agent_core.py


In [10]:
%%writefile test_energy_agent.py

import json
import pytest
from pydantic import ValidationError

from energy_agent_core import (
    EnergyStatusInput,
    SafeLoadInput,
    BatteryCheckInput,
    SetPowerLimitInput,
    get_energy_status,
    calculate_safe_load,
    check_battery_safety,
)


def test_valid_inverter_id():
    model = EnergyStatusInput(
        inverter_id="inverter_1"
    )
    assert model.inverter_id == "inverter_1"


def test_invalid_inverter_id():
    with pytest.raises(ValidationError):
        EnergyStatusInput(
            inverter_id="device_1"
        )


def test_valid_reserve():
    model = SafeLoadInput(
        pv_power_w=4200,
        current_load_w=2800,
        reserve_w=500
    )
    assert model.reserve_w == 500


def test_invalid_reserve():
    with pytest.raises(ValidationError):
        SafeLoadInput(
            pv_power_w=4200,
            current_load_w=2800,
            reserve_w=525
        )


def test_invalid_soc():
    with pytest.raises(ValidationError):
        BatteryCheckInput(
            soc=120,
            minimum_soc=20
        )


def test_invalid_minimum_soc():
    with pytest.raises(ValidationError):
        BatteryCheckInput(
            soc=70,
            minimum_soc=5
        )


def test_invalid_power_limit():
    with pytest.raises(ValidationError):
        SetPowerLimitInput(
            inverter_id="inverter_1",
            limit_w=5555,
            reason="Некоректний тестовий ліміт"
        )


def test_get_energy_status_tool():
    result = json.loads(
        get_energy_status.invoke(
            {"inverter_id": "inverter_1"}
        )
    )

    assert result["status"] == "success"
    assert "pv_power_w" in result["data"]


def test_calculate_safe_load_tool():
    result = json.loads(
        calculate_safe_load.invoke(
            {
                "pv_power_w": 4200,
                "current_load_w": 2800,
                "reserve_w": 500
            }
        )
    )

    assert result["status"] == "success"
    assert (
        result["data"]["available_power_w"]
        == 900
    )


def test_check_battery_safety_tool():
    result = json.loads(
        check_battery_safety.invoke(
            {
                "soc": 72,
                "minimum_soc": 20
            }
        )
    )

    assert result["status"] == "success"
    assert (
        result["data"]["discharge_allowed"]
        is True
    )

Overwriting test_energy_agent.py


In [11]:
!pytest -v test_energy_agent.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: asyncio-1.4.0, langsmith-0.11.0, anyio-4.14.2, typeguard-4.6.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 10 items                                                             

test_energy_agent.py::test_valid_inverter_id PASSED                      [ 10%]
test_energy_agent.py::test_invalid_inverter_id PASSED                    [ 20%]
test_energy_agent.py::test_valid_reserve PASSED                          [ 30%]
test_energy_agent.py::test_invalid_reserve PASSED                        [ 40%]
test_energy_agent.py::test_invalid_soc PASSED                            [ 50%]
test_energy_agent.py::test_invalid_minimum_soc PASSED                    [ 60%]
test_energy_agent.py::test_invalid_power_

In [13]:
# ============================================================
# КРОК 5. Налаштування Gemini
# ============================================================

import os
import time
from google.colab import userdata

SECRET_NAME = "GOOGLE_API_KEY"

for attempt in range(3):
    try:
        GOOGLE_API_KEY = userdata.get(SECRET_NAME)

        if GOOGLE_API_KEY:
            os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
            print("✅ GOOGLE_API_KEY завантажено")
            break

    except Exception as e:
        print(f"⚠️ Спроба {attempt + 1}/3: {type(e).__name__}")
        time.sleep(2)

else:
    raise RuntimeError(
        "❌ Не вдалося прочитати GOOGLE_API_KEY з Colab Secrets. "
        "Перевір назву секрету та доступ до записника."
    )

✅ GOOGLE_API_KEY завантажено


In [14]:
# ============================================================
# КРОК 6. Створення Gemini-моделі для ReAct-агента
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

print("✅ Gemini model створено")

✅ Gemini model створено


In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("✅ Gemini model створено")

✅ Gemini model створено


In [20]:
# ============================================================
# КРОК 7. ReAct-агент у LangGraph
# LLM -> tools -> LLM
# max_steps=10, repeat detection, JSON trajectory
# ============================================================

import json
import time
from typing import Annotated, TypedDict

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from energy_agent_core import (
    get_energy_status,
    calculate_safe_load,
    check_battery_safety,
    set_inverter_power_limit,
)


# ------------------------------------------------------------
# Інструменти агента
# ------------------------------------------------------------

react_tools = [
    get_energy_status,
    calculate_safe_load,
    check_battery_safety,
    set_inverter_power_limit,
]

tools_by_name = {
    tool.name: tool
    for tool in react_tools
}

# Передаємо опис tools у Gemini
llm_with_tools = llm.bind_tools(react_tools)


# ------------------------------------------------------------
# Системний prompt
# ------------------------------------------------------------

SYSTEM_PROMPT = """
Ти Energy Management Agent.

Твоє завдання:
- аналізувати стан домашньої енергосистеми;
- перевіряти PV generation, load та SOC батареї;
- використовувати tools тільки коли це потрібно;
- не вигадувати значення, які можна отримати через tool;
- ризиковий tool set_inverter_power_limit не викликати без
  явної необхідності зміни ліміту.

Працюй коротко та послідовно.
Якщо для відповіді достатньо отриманих даних — дай фінальну відповідь.
"""


# ------------------------------------------------------------
# State LangGraph
# ------------------------------------------------------------

class ReActState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

    # Кількість виконаних кроків графа
    step_count: int

    # Історія tool-викликів для детекції повторів
    tool_history: list[str]

    # JSON trajectory
    trajectory: list[dict]

    # Час старту
    started_at: float

    # Причина аварійного завершення
    stop_reason: str | None


MAX_STEPS = 10
TIMEOUT_SECONDS = 120


# ------------------------------------------------------------
# Допоміжна функція логування
# ------------------------------------------------------------

def add_trajectory_event(
    state: ReActState,
    event_type: str,
    data: dict,
) -> list[dict]:

    trajectory = list(state.get("trajectory", []))

    trajectory.append({
        "timestamp": time.time(),
        "step": state.get("step_count", 0),
        "event": event_type,
        "data": data,
    })

    return trajectory


# ------------------------------------------------------------
# NODE 1: LLM
# ------------------------------------------------------------

def llm_node(state: ReActState):

    elapsed = time.time() - state["started_at"]

    # Timeout protection
    if elapsed > TIMEOUT_SECONDS:
        return {
            "stop_reason": "timeout",
            "trajectory": add_trajectory_event(
                state,
                "stop",
                {
                    "reason": "timeout",
                    "elapsed_seconds": elapsed,
                }
            ),
        }

    # Max steps protection
    if state["step_count"] >= MAX_STEPS:
        return {
            "stop_reason": "max_steps",
            "trajectory": add_trajectory_event(
                state,
                "stop",
                {
                    "reason": "max_steps",
                    "max_steps": MAX_STEPS,
                }
            ),
        }

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"],
    ]

    response = llm_with_tools.invoke(messages)

    trajectory = add_trajectory_event(
        state,
        "llm_response",
        {
            "content": response.content,
            "tool_calls": response.tool_calls,
        }
    )

    return {
        "messages": [response],
        "step_count": state["step_count"] + 1,
        "trajectory": trajectory,
    }


# ------------------------------------------------------------
# NODE 2: TOOLS
# ------------------------------------------------------------

def tools_node(state: ReActState):

    last_message = state["messages"][-1]

    new_messages = []
    history = list(state.get("tool_history", []))
    trajectory = list(state.get("trajectory", []))

    for call in last_message.tool_calls:

        tool_name = call["name"]
        arguments = call["args"]

        # Створюємо стабільний ключ виклику
        call_signature = json.dumps(
            {
                "tool": tool_name,
                "args": arguments,
            },
            sort_keys=True,
            ensure_ascii=False,
        )

        # ----------------------------------------------------
        # Repeat detection
        # ----------------------------------------------------

        if call_signature in history:

            result = json.dumps(
                {
                    "status": "error",
                    "error": "Повторний ідентичний tool-виклик заблоковано",
                },
                ensure_ascii=False,
            )

            trajectory.append({
                "timestamp": time.time(),
                "step": state["step_count"],
                "event": "repeat_blocked",
                "data": {
                    "tool": tool_name,
                    "args": arguments,
                },
            })

        else:

            history.append(call_signature)

            try:
                selected_tool = tools_by_name[tool_name]
                result = selected_tool.invoke(arguments)

            except Exception as exc:
                result = json.dumps(
                    {
                        "status": "error",
                        "error": str(exc),
                    },
                    ensure_ascii=False,
                )

            trajectory.append({
                "timestamp": time.time(),
                "step": state["step_count"],
                "event": "tool_call",
                "data": {
                    "tool": tool_name,
                    "args": arguments,
                    "result": result,
                },
            })

        new_messages.append(
            ToolMessage(
                content=result,
                tool_call_id=call["id"],
            )
        )

    return {
        "messages": new_messages,
        "tool_history": history,
        "trajectory": trajectory,
        "step_count": state["step_count"] + 1,
    }


# ------------------------------------------------------------
# ROUTER
# ------------------------------------------------------------

def route_after_llm(state: ReActState):

    if state.get("stop_reason"):
        return END

    last_message = state["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        return "tools"

    return END


# ------------------------------------------------------------
# Побудова LangGraph
# ------------------------------------------------------------

builder = StateGraph(ReActState)

builder.add_node("llm", llm_node)
builder.add_node("tools", tools_node)

builder.add_edge(START, "llm")

builder.add_conditional_edges(
    "llm",
    route_after_llm,
    {
        "tools": "tools",
        END: END,
    },
)

builder.add_edge("tools", "llm")

react_graph = builder.compile()

print("✅ ReAct LangGraph створено")
print("✅ max_steps =", MAX_STEPS)
print("✅ timeout =", TIMEOUT_SECONDS, "сек.")
print("✅ repeat detection = ON")
print("✅ trajectory logging = ON")

✅ ReAct LangGraph створено
✅ max_steps = 10
✅ timeout = 120 сек.
✅ repeat detection = ON
✅ trajectory logging = ON


In [21]:
# ============================================================
# КРОК 8. Демонстраційний запуск ReAct-агента
# ============================================================

import json
import time
from langchain_core.messages import HumanMessage

initial_state = {
    "messages": [
        HumanMessage(
            content=(
                "Перевір inverter_1 і скажи, "
                "скільки додаткового навантаження можна "
                "безпечно підключити при резерві 500 Вт."
            )
        )
    ],
    "step_count": 0,
    "tool_history": [],
    "trajectory": [],
    "started_at": time.time(),
    "stop_reason": None,
}

result = react_graph.invoke(
    initial_state,
    config={"recursion_limit": 20}
)

print("=== ФІНАЛЬНА ВІДПОВІДЬ ===")
print(result["messages"][-1].content)

print("\n=== КІЛЬКІСТЬ КРОКІВ ===")
print(result["step_count"])

print("\n=== TOOL HISTORY ===")
for item in result["tool_history"]:
    print(item)

print("\n=== TRAJECTORY ===")
print(
    json.dumps(
        result["trajectory"],
        ensure_ascii=False,
        indent=2
    )
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== ФІНАЛЬНА ВІДПОВІДЬ ===
[{'type': 'text', 'text': 'За результатами перевірки `inverter_1`:\n\n* **Генерація сонячних панелей (PV):** 4200 Вт\n* **Поточне навантаження:** 2800 Вт\n* **Заряд батареї (SOC):** 72%\n* **Необхідний резерв:** 500 Вт\n\n**Додаткове безпечне навантаження:** **900 Вт** *(4200 Вт − 2800 Вт − 500 Вт)*.', 'extras': {'signature': 'EpYDCpMDARFNMg+geNiI3PnKptM/6KNKwMsfQM68TN2XqySHiUSxy9sYkru0NePR5UMgmTYdmMGgF3y+eJ3mBz7ChJYW8JfLXVYsiuENhoYHBr2ncZoiNr7DJBnoqJDOYvjJKLHbe/151RAx5SvJt3uo0BfTWO8TaRXMiK+o1Ju0fPccQ1thI1QsfLV2wUGAbMdlIPwOs1rCuCxuu9OSMclKvaB3XpEioSqqKumRyaYp+6jClK0IkdMbdG6/8MTDIcKBOu2zRrOkV62kX7iCNn/VNZbA8ixBS4Xw4+3ea7HKOLwZjAn3LgQghj6iMDL0tVkBr/VQm53syZJFgrU7UqCCKRSMjr3aYWVdJ/zZ6A4NxotA+TwOKBNqLSWuJw0+wL/LdyXRqsqXSsOe4FIBV0WqlLjGf79vqREo0HOkMRTblBWiOhX8x0V75/y4ZwaCjgdm1tvjyX/8v7+YX2j0IePBHw1zv+BN43m9cKaVRJ/l1v3SEIaMIGij6geyr06cNcWivePBN88rOem/lw3wsA+zfZnUxUsXuQ=='}}]

=== КІЛЬКІСТЬ КРОКІВ ===
5

=== TOOL HISTORY ===
{"args": {"inverter_id": "inverter_1"}, "

In [22]:
# ============================================================
# КРОК 9. Збереження JSON-логу траєкторії
# ============================================================

import json

with open("trajectory.json", "w", encoding="utf-8") as f:
    json.dump(
        result["trajectory"],
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ trajectory.json збережено")
print("Кількість подій:", len(result["trajectory"]))

✅ trajectory.json збережено
Кількість подій: 5


In [23]:
!ls -lh trajectory.json

-rw-r--r-- 1 root root 2.8K Aug 24 07:44 trajectory.json


In [24]:
# ============================================================
# КРОК 10. Pydantic-моделі для Plan-and-Execute
# ============================================================

from pydantic import BaseModel, Field
from typing import List, Literal, Optional


class Plan(BaseModel):
    """Структурований план виконання складної задачі."""

    steps: List[str] = Field(
        ...,
        min_length=1,
        max_length=6,
        description="Послідовність коротких кроків для виконання задачі"
    )


class ReplanDecision(BaseModel):
    """Рішення replanner: завершити роботу або оновити план."""

    action: Literal["continue", "finish"] = Field(
        ...,
        description="continue — продовжити, finish — завершити"
    )

    remaining_steps: List[str] = Field(
        default_factory=list,
        description="Оновлений список невиконаних кроків"
    )

    final_answer: Optional[str] = Field(
        default=None,
        description="Фінальна відповідь, якщо action=finish"
    )


print("✅ Plan model створено")
print("✅ ReplanDecision model створено")

✅ Plan model створено
✅ ReplanDecision model створено


In [27]:
# ============================================================
# КРОК 11. Plan-and-Execute агент
# planner -> executor -> replanner
# ============================================================

from typing import TypedDict
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END


# ------------------------------------------------------------
# Structured output моделі
# ------------------------------------------------------------

planner_llm = llm.with_structured_output(Plan)
replanner_llm = llm.with_structured_output(ReplanDecision)


# ------------------------------------------------------------
# State
# ------------------------------------------------------------

class PlanExecuteState(TypedDict):
    task: str
    plan: list[str]
    completed_steps: list[str]
    current_result: str
    final_answer: str | None


# ------------------------------------------------------------
# NODE 1: PLANNER
# ------------------------------------------------------------

def planner_node(state: PlanExecuteState):

    prompt = f"""

Ти planner для Energy Management Agent.

Задача користувача:
{state["task"]}

Сформуй короткий план із 2-4 кроків.

ВАЖЛИВО:
- для визначення поточного стану використовуй get_energy_status;
- для розрахунку безпечного додаткового навантаження
  ОБОВ'ЯЗКОВО використовуй calculate_safe_load;
- не розраховуй доступне навантаження через power_limit_w;
- доступна потужність рахується від поточної PV generation;
- не виконуй ризикові операції без необхідності.

План має бути конкретним і без зайвих кроків.
"""

    plan_result = planner_llm.invoke(prompt)

    return {
        "plan": plan_result.steps,
        "completed_steps": [],
    }


# ------------------------------------------------------------
# NODE 2: EXECUTOR
# Виконуємо один крок через наш ReAct-агент
# ------------------------------------------------------------

def executor_node(state: PlanExecuteState):

    if not state["plan"]:
        return {
            "current_result": "Немає кроків для виконання"
        }

    current_step = state["plan"][0]

    react_state = {
        "messages": [
            HumanMessage(
                content=f"""
Виконай тільки цей крок плану:

{current_step}

Основна задача:
{state["task"]}
"""
            )
        ],
        "step_count": 0,
        "tool_history": [],
        "trajectory": [],
        "started_at": time.time(),
        "stop_reason": None,
    }

    react_result = react_graph.invoke(
        react_state,
        config={"recursion_limit": 20}
    )

    step_result = react_result["messages"][-1].content

    completed = list(state.get("completed_steps", []))
    completed.append(current_step)

    remaining = list(state["plan"][1:])

    return {
        "plan": remaining,
        "completed_steps": completed,
        "current_result": step_result,
    }


# ------------------------------------------------------------
# NODE 3: REPLANNER
# ------------------------------------------------------------

def replanner_node(state: PlanExecuteState):

    prompt = f"""
Ти replanner Energy Management Agent.

Початкова задача:
{state["task"]}

Виконані кроки:
{state["completed_steps"]}

Останній результат:
{state["current_result"]}

Кроки, що залишились:
{state["plan"]}

Прийми рішення:

- finish, якщо інформації вже достатньо;
- continue, якщо потрібно виконати ще кроки.

Якщо finish:
обов'язково сформуй final_answer.

Якщо continue:
поверни remaining_steps.
"""

    decision = replanner_llm.invoke(prompt)

    if decision.action == "finish":
        return {
            "final_answer": decision.final_answer,
            "plan": [],
        }

    return {
        "plan": decision.remaining_steps,
    }


# ------------------------------------------------------------
# Router
# ------------------------------------------------------------

def route_after_replanner(state: PlanExecuteState):

    if state.get("final_answer"):
        return END

    if not state.get("plan"):
        return END

    return "executor"


# ------------------------------------------------------------
# Побудова графа
# ------------------------------------------------------------

plan_builder = StateGraph(PlanExecuteState)

plan_builder.add_node("planner", planner_node)
plan_builder.add_node("executor", executor_node)
plan_builder.add_node("replanner", replanner_node)

plan_builder.add_edge(START, "planner")
plan_builder.add_edge("planner", "executor")
plan_builder.add_edge("executor", "replanner")

plan_builder.add_conditional_edges(
    "replanner",
    route_after_replanner,
    {
        "executor": "executor",
        END: END,
    },
)

plan_execute_graph = plan_builder.compile()

print("✅ Plan-and-Execute LangGraph створено")
print("✅ planner -> executor -> replanner")
print("✅ Plan structured output = ON")
print("✅ ReplanDecision structured output = ON")

✅ Plan-and-Execute LangGraph створено
✅ planner -> executor -> replanner
✅ Plan structured output = ON
✅ ReplanDecision structured output = ON


In [28]:
# ============================================================
# КРОК 12. Демонстрація Plan-and-Execute
# ============================================================

plan_result = plan_execute_graph.invoke(
    {
        "task": (
            "Перевір inverter_1 і визнач, "
            "скільки додаткового навантаження можна "
            "безпечно підключити при резерві 500 Вт."
        ),
        "plan": [],
        "completed_steps": [],
        "current_result": "",
        "final_answer": None,
    },
    config={"recursion_limit": 20}
)

print("=== PLAN-AND-EXECUTE RESULT ===")

print("\nВиконані кроки:")
for i, step in enumerate(plan_result["completed_steps"], start=1):
    print(f"{i}. {step}")

print("\nОстанній результат:")
print(plan_result["current_result"])

print("\nФінальна відповідь:")
print(plan_result["final_answer"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/l

=== PLAN-AND-EXECUTE RESULT ===

Виконані кроки:
1. Отримати поточний стан інвертора inverter_1 та PV генерації за допомогою get_energy_status.
2. Розрахувати безпечне додаткове навантаження з резервом 500 Вт за допомогою calculate_safe_load.

Останній результат:
[{'type': 'text', 'text': 'За результатами перевірки `inverter_1`:\n- Поточна генерація PV (`pv_power_w`): 4200 Вт\n- Поточне навантаження (`current_load_w`): 2800 Вт\n- Резерв потужності (`reserve_w`): 500 Вт\n\n**Результат розрахунку:**\nМожна безпечно підключити додаткове навантаження до **900 Вт** (`can_add_load`: true).', 'extras': {'signature': 'EogCCoUCARFNMg/qzlyDQsGH4oD9FY4jOSQ1683cPrXs8ApxVLdXmMfuTuzYQs9z1Oc0L0MFmlRejtHuW+PU6jdfQZZonwec5t2MuFGi8U3VK/FLmHxc4ExJvsuX2MoOe5/g1kkOdnuIQWq5kaKaCc7j51XxF2WLIQ9lDW5rsRU4DbUeE3X63CAmHxvtNL0ainzuIfWKFCxbxeAv29RVs9R/RBqeMFYqfFamJDqvHH5fHrUgDKxwgqM5eNAXEGEq4x6ElXZnw9oiytnbpMXpGhiURX73if3h5F5/O762rfuZMfdmIGkKWB3zVPGIK8J8vGIbnje10vM0e1zMN/Yw0xLTw8Ju8ThV13lP'}}]

Фінальна відповідь:


In [30]:
!pip install -q langgraph-checkpoint-sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 11.1 MB/s eta 0:00:00


In [31]:
# ============================================================
# КРОК 13. Checkpointer через SqliteSaver
# ============================================================

import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver

# Створюємо SQLite-файл для збереження стану
conn = sqlite3.connect(
    "energy_agent_checkpoints.sqlite",
    check_same_thread=False
)

checkpointer = SqliteSaver(conn)

print("✅ SqliteSaver створено")
print("✅ Файл: energy_agent_checkpoints.sqlite")

✅ SqliteSaver створено
✅ Файл: energy_agent_checkpoints.sqlite


In [32]:
# ============================================================
# КРОК 14. Демонстрація checkpointing та відновлення
# ============================================================

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class CheckpointState(TypedDict):
    value: int
    history: list[str]


def step_one(state: CheckpointState):
    history = list(state.get("history", []))
    history.append("step_one виконано")

    return {
        "value": state["value"] + 1,
        "history": history
    }


def step_two(state: CheckpointState):
    history = list(state.get("history", []))
    history.append("step_two виконано")

    return {
        "value": state["value"] * 2,
        "history": history
    }


checkpoint_builder = StateGraph(CheckpointState)

checkpoint_builder.add_node("step_one", step_one)
checkpoint_builder.add_node("step_two", step_two)

checkpoint_builder.add_edge(START, "step_one")
checkpoint_builder.add_edge("step_one", "step_two")
checkpoint_builder.add_edge("step_two", END)

checkpoint_graph = checkpoint_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["step_two"]
)

print("✅ Checkpoint graph створено")
print("✅ interrupt_before=['step_two']")

✅ Checkpoint graph створено
✅ interrupt_before=['step_two']


In [33]:
# ============================================================
# КРОК 15. Запуск до interrupt + get_state()
# ============================================================

config = {
    "configurable": {
        "thread_id": "energy_checkpoint_demo"
    }
}

first_run = checkpoint_graph.invoke(
    {
        "value": 10,
        "history": []
    },
    config=config
)

print("=== СТАН ПІСЛЯ ПЕРЕРИВАННЯ ===")
print(first_run)

snapshot = checkpoint_graph.get_state(config)

print("\n=== GET_STATE() ===")
print("values:", snapshot.values)
print("next:", snapshot.next)

=== СТАН ПІСЛЯ ПЕРЕРИВАННЯ ===
{'value': 11, 'history': ['step_one виконано']}

=== GET_STATE() ===
values: {'value': 11, 'history': ['step_one виконано']}
next: ('step_two',)


In [34]:
# ============================================================
# КРОК 16. Відновлення після переривання
# ============================================================

resumed = checkpoint_graph.invoke(
    None,
    config=config
)

print("=== СТАН ПІСЛЯ ВІДНОВЛЕННЯ ===")
print(resumed)

final_snapshot = checkpoint_graph.get_state(config)

print("\n=== ФІНАЛЬНИЙ SNAPSHOT ===")
print("values:", final_snapshot.values)
print("next:", final_snapshot.next)

=== СТАН ПІСЛЯ ВІДНОВЛЕННЯ ===
{'value': 22, 'history': ['step_one виконано', 'step_two виконано']}

=== ФІНАЛЬНИЙ SNAPSHOT ===
values: {'value': 22, 'history': ['step_one виконано', 'step_two виконано']}
next: ()


In [35]:
# ============================================================
# КРОК 17. База знань для Agentic RAG
# ============================================================

knowledge_docs = [
    """
    Для безпечної роботи домашньої енергосистеми бажано
    залишати резерв потужності щонайменше 500 Вт.
    """,

    """
    Додаткове навантаження слід підключати лише тоді,
    коли поточна PV-генерація перевищує поточне навантаження
    та заданий резерв.
    """,

    """
    Рекомендований мінімальний SOC батареї для щоденної роботи
    становить 20%. Нижче цього рівня розряд бажано обмежувати.
    """,

    """
    Для LiFePO4 батарей небажано регулярно допускати
    глибокий розряд до 0%, оскільки це може скорочувати ресурс.
    """,

    """
    При нестабільній мережевій напрузі необхідно перевірити
    якість контактів, кабельні з'єднання та захисну автоматику.
    """,

    """
    Потужність сонячної генерації залежить від освітленості,
    температури панелей, орієнтації та затінення.
    """,

    """
    Зміну максимального power limit інвертора слід вважати
    ризиковою операцією та виконувати лише після підтвердження
    користувачем.
    """,

    """
    Якщо даних для прийняття рішення недостатньо,
    агент повинен спочатку отримати фактичний стан системи,
    а не робити припущення.
    """,

    """
    Для оцінки доступної додаткової потужності використовується
    формула:
    available_power = pv_power - current_load - reserve.
    """,

    """
    У критичних енергетичних сценаріях бажано використовувати
    human-in-the-loop перед зміною параметрів інвертора.
    """
]

print("✅ Документів у базі знань:", len(knowledge_docs))

✅ Документів у базі знань: 10


In [36]:
# ============================================================
# КРОК 18. ChromaDB + RAG tool
# ============================================================

import json

from pydantic import BaseModel, Field, field_validator
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma


# ------------------------------------------------------------
# Embeddings
# ------------------------------------------------------------

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)


# ------------------------------------------------------------
# Перетворюємо тексти у Document
# ------------------------------------------------------------

documents = [
    Document(
        page_content=text.strip(),
        metadata={"source": f"energy_rule_{i+1}"}
    )
    for i, text in enumerate(knowledge_docs)
]


# ------------------------------------------------------------
# Створюємо ChromaDB
# ------------------------------------------------------------

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="energy_knowledge_base"
)

print("✅ ChromaDB створено")
print("✅ Документів:", len(documents))


# ============================================================
# Pydantic-схема RAG tool
# ============================================================

class RAGSearchInput(BaseModel):

    query: str = Field(
        ...,
        min_length=3,
        max_length=300,
        description="Питання для пошуку у базі знань"
    )

    top_k: int = Field(
        default=3,
        ge=1,
        le=5,
        description="Кількість релевантних документів"
    )

    @field_validator("query")
    @classmethod
    def validate_query(cls, value: str) -> str:

        value = value.strip()

        if not value:
            raise ValueError(
                "Пошуковий запит не може бути порожнім"
            )

        return value


# ============================================================
# RAG TOOL
# ============================================================

@tool(args_schema=RAGSearchInput)
def search_energy_knowledge(
    query: str,
    top_k: int = 3
) -> str:
    """
    Шукає правила та рекомендації у базі знань
    домашньої енергосистеми.
    Використовуй цей tool, коли потрібні правила,
    рекомендації або довідкова інформація.
    """

    try:

        results = vectorstore.similarity_search(
            query,
            k=top_k
        )

        data = [
            {
                "content": doc.page_content,
                "source": doc.metadata.get("source")
            }
            for doc in results
        ]

        return json.dumps(
            {
                "status": "success",
                "data": data
            },
            ensure_ascii=False
        )

    except Exception as exc:

        return json.dumps(
            {
                "status": "error",
                "error": str(exc)
            },
            ensure_ascii=False
        )


print("✅ RAG tool створено:", search_energy_knowledge.name)

✅ ChromaDB створено
✅ Документів: 10
✅ RAG tool створено: search_energy_knowledge


In [37]:
# ============================================================
# КРОК 19. Локальна перевірка RAG tool
# ============================================================

rag_test = search_energy_knowledge.invoke(
    {
        "query": "Який мінімальний SOC батареї рекомендується?",
        "top_k": 3
    }
)

print("=== RAG RESULT ===")
print(rag_test)

=== RAG RESULT ===
{"status": "success", "data": [{"content": "Рекомендований мінімальний SOC батареї для щоденної роботи\n    становить 20%. Нижче цього рівня розряд бажано обмежувати.", "source": "energy_rule_3"}, {"content": "Для LiFePO4 батарей небажано регулярно допускати\n    глибокий розряд до 0%, оскільки це може скорочувати ресурс.", "source": "energy_rule_4"}, {"content": "Додаткове навантаження слід підключати лише тоді,\n    коли поточна PV-генерація перевищує поточне навантаження\n    та заданий резерв.", "source": "energy_rule_2"}]}


In [38]:
# ============================================================
# КРОК 20. Agentic RAG: додаємо RAG tool до ReAct-агента
# ============================================================

# Додаємо RAG tool до списку інструментів
react_tools_with_rag = [
    get_energy_status,
    calculate_safe_load,
    check_battery_safety,
    set_inverter_power_limit,
    search_energy_knowledge,
]

tools_by_name = {
    tool.name: tool
    for tool in react_tools_with_rag
}

# Переприв'язуємо tools до Gemini
llm_with_tools = llm.bind_tools(react_tools_with_rag)

# Оновлюємо системний prompt
SYSTEM_PROMPT = """
Ти Energy Management Agent.

Твоє завдання:
- аналізувати стан домашньої енергосистеми;
- перевіряти PV generation, load та SOC батареї;
- використовувати tools тільки коли це потрібно;
- не вигадувати значення, які можна отримати через tool;
- якщо користувач питає про правила, рекомендації,
  нормативи або best practices — використовуй
  search_energy_knowledge;
- для фактичних поточних даних використовуй
  get_energy_status;
- ризиковий tool set_inverter_power_limit не викликати
  без явної необхідності.

Якщо для відповіді достатньо отриманих даних —
дай фінальну відповідь.
"""

print("✅ RAG tool додано до ReAct-агента")
print("✅ Кількість tools:", len(react_tools_with_rag))

✅ RAG tool додано до ReAct-агента
✅ Кількість tools: 5


In [39]:
# ============================================================
# КРОК 21. Демонстрація Agentic RAG
# ============================================================

rag_agent_state = {
    "messages": [
        HumanMessage(
            content=(
                "Який мінімальний SOC батареї рекомендується "
                "для щоденної роботи і що робити нижче цього рівня?"
            )
        )
    ],
    "step_count": 0,
    "tool_history": [],
    "trajectory": [],
    "started_at": time.time(),
    "stop_reason": None,
}

rag_agent_result = react_graph.invoke(
    rag_agent_state,
    config={"recursion_limit": 20}
)

print("=== AGENTIC RAG RESULT ===")
print(rag_agent_result["messages"][-1].content)

print("\n=== TOOL HISTORY ===")
for item in rag_agent_result["tool_history"]:
    print(item)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== AGENTIC RAG RESULT ===
[{'type': 'text', 'text': 'Згідно з рекомендаціями базі знань енергосистеми:\n\n1. **Мінімальний рекомендований SOC для щоденної роботи:** **20%**.\n2. **Що робити, якщо рівень заряду впав нижче 20%:**\n   * **Обмежувати розряд:** небажано продовжувати активний розряд акумулятора.\n   * **Уникати глибокого розряду (до 0%):** особливо для LiFePO4 акумуляторів регулярний глибокий розряд скорочує термін служби та ресурс батареї.\n   * Рекомендується зменшити споживання електроенергії або підключити зовнішнє джерело живлення/мережу для підзарядки.', 'extras': {'signature': 'EvwDCvkDARFNMg8fzf4tphfbU3wRmz+7wd+zSD+2bz789KZZUvoBrWNDNhpBfxym091YD8ycD/Xje2ufPNv4T5KAqkUoxWTtKnGcNmrq2KWRok/FXJAW/vfZ1Q7/0aPWsTHZhMz/rLcWs0ZztK6VKIaYdyKK/opYsnPrmb69Mj0BD1Cn0rjws7bkSqIOO+0uvCO3n4fyzap1ths9JNfZI8s586gRAFkzV4lcPa1qTXxgHH++i2Gax0Z0b8eJEgTmPA3zonZcoAX2oM/pDHuxF9QKB9gWskdS/rOHbR0iY64Pvuke852NkrIKZSSOOC2+15q3ahDWApRW3wQPK0bXG1WRJ/78lbLLhNB/bFDTpfa0BHE/hUdn1wCQGntmt10MPJL0yd9biZQW

In [40]:
# ============================================================
# КРОК 22. Human-in-the-Loop для ризикової операції
# ============================================================

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class HITLState(TypedDict):
    inverter_id: str
    requested_limit_w: int
    reason: str
    approved: bool | None
    result: str | None


def prepare_change(state: HITLState):
    """
    Підготовка ризикової операції.
    Після цього граф буде зупинено перед виконанням.
    """
    return {
        "approved": None,
        "result": None
    }


def execute_change(state: HITLState):
    """
    Виконує зміну тільки якщо користувач підтвердив операцію.
    """

    if state["approved"] is not True:
        return {
            "result": json.dumps(
                {
                    "status": "error",
                    "error": "Операцію відхилено користувачем"
                },
                ensure_ascii=False
            )
        }

    result = set_inverter_power_limit.invoke(
        {
            "inverter_id": state["inverter_id"],
            "limit_w": state["requested_limit_w"],
            "reason": state["reason"]
        }
    )

    return {
        "result": result
    }


hitl_builder = StateGraph(HITLState)

hitl_builder.add_node("prepare_change", prepare_change)
hitl_builder.add_node("execute_change", execute_change)

hitl_builder.add_edge(START, "prepare_change")
hitl_builder.add_edge("prepare_change", "execute_change")
hitl_builder.add_edge("execute_change", END)

hitl_graph = hitl_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute_change"]
)

print("✅ HITL graph створено")
print("✅ interrupt_before=['execute_change']")

✅ HITL graph створено
✅ interrupt_before=['execute_change']


In [41]:
# ============================================================
# КРОК 23. HITL — APPROVE
# ============================================================

approve_config = {
    "configurable": {
        "thread_id": "hitl_approve_demo"
    }
}

approve_start = hitl_graph.invoke(
    {
        "inverter_id": "inverter_1",
        "requested_limit_w": 5000,
        "reason": "Демонстрація Human-in-the-Loop",
        "approved": None,
        "result": None
    },
    config=approve_config
)

print("=== СТАН ДО ПІДТВЕРДЖЕННЯ ===")
print(approve_start)

approve_snapshot = hitl_graph.get_state(approve_config)

print("\nnext:", approve_snapshot.next)

=== СТАН ДО ПІДТВЕРДЖЕННЯ ===
{'inverter_id': 'inverter_1', 'requested_limit_w': 5000, 'reason': 'Демонстрація Human-in-the-Loop', 'approved': None, 'result': None}

next: ('execute_change',)


In [42]:
# ============================================================
# КРОК 24. Підтвердження операції
# ============================================================

hitl_graph.update_state(
    approve_config,
    {"approved": True}
)

approved_result = hitl_graph.invoke(
    None,
    config=approve_config
)

print("=== APPROVED RESULT ===")
print(approved_result["result"])

=== APPROVED RESULT ===
{"status": "success", "data": {"old_limit_w": 6000, "new_limit_w": 5000, "reason": "Демонстрація Human-in-the-Loop"}}


In [43]:
# ============================================================
# КРОК 25. HITL — REJECT
# ============================================================

reject_config = {
    "configurable": {
        "thread_id": "hitl_reject_demo"
    }
}

reject_start = hitl_graph.invoke(
    {
        "inverter_id": "inverter_1",
        "requested_limit_w": 4000,
        "reason": "Демонстрація відхилення Human-in-the-Loop",
        "approved": None,
        "result": None
    },
    config=reject_config
)

print("=== СТАН ДО РІШЕННЯ ===")
print(reject_start)

reject_snapshot = hitl_graph.get_state(reject_config)

print("\nnext:", reject_snapshot.next)

=== СТАН ДО РІШЕННЯ ===
{'inverter_id': 'inverter_1', 'requested_limit_w': 4000, 'reason': 'Демонстрація відхилення Human-in-the-Loop', 'approved': None, 'result': None}

next: ('execute_change',)


In [44]:
# ============================================================
# КРОК 26. Відхилення операції
# ============================================================

hitl_graph.update_state(
    reject_config,
    {"approved": False}
)

rejected_result = hitl_graph.invoke(
    None,
    config=reject_config
)

print("=== REJECTED RESULT ===")
print(rejected_result["result"])

=== REJECTED RESULT ===
{"status": "error", "error": "Операцію відхилено користувачем"}


In [45]:
print(
    get_energy_status.invoke(
        {"inverter_id": "inverter_1"}
    )
)

{"status": "success", "data": {"pv_power_w": 4200, "load_power_w": 2800, "battery_soc": 72, "battery_power_w": -900, "grid_power_w": -500, "power_limit_w": 5000}}


In [46]:
# ============================================================
# КРОК 27. Базовий тест ReAct-циклу
# ============================================================

%%writefile -a test_energy_agent.py

def test_basic_react_components():
    """
    Базова перевірка компонентів ReAct-агента без виклику Gemini.
    Перевіряємо, що основні tools доступні та мають коректні імена.
    """

    from energy_agent_core import (
        get_energy_status,
        calculate_safe_load,
        check_battery_safety,
        set_inverter_power_limit,
    )

    tools = [
        get_energy_status,
        calculate_safe_load,
        check_battery_safety,
        set_inverter_power_limit,
    ]

    names = [tool.name for tool in tools]

    assert "get_energy_status" in names
    assert "calculate_safe_load" in names
    assert "check_battery_safety" in names
    assert "set_inverter_power_limit" in names

Appending to test_energy_agent.py


In [47]:
!pytest -v test_energy_agent.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: asyncio-1.4.0, langsmith-0.11.0, anyio-4.14.2, typeguard-4.6.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 11 items                                                             

test_energy_agent.py::test_valid_inverter_id PASSED                      [  9%]
test_energy_agent.py::test_invalid_inverter_id PASSED                    [ 18%]
test_energy_agent.py::test_valid_reserve PASSED                          [ 27%]
test_energy_agent.py::test_invalid_reserve PASSED                        [ 36%]
test_energy_agent.py::test_invalid_soc PASSED                            [ 45%]
test_energy_agent.py::test_invalid_minimum_soc PASSED                    [ 54%]
test_energy_agent.py::test_invalid_power_

In [50]:
# ============================================================
# ФІНАЛ. Пакуємо всі файли проєкту в ZIP
# ============================================================

import shutil
import os

project_files = [
    "Task_001_Бабенко_EnergyAgent.ipynb",
    "energy_agent_core.py",
    "test_energy_agent.py",
    "trajectory.json",
    "energy_agent_checkpoints.sqlite",
    "README.md",
]

os.makedirs("EnergyAgent_project", exist_ok=True)

for file_name in project_files:
    if os.path.exists(file_name):
        shutil.copy(file_name, "EnergyAgent_project/")
        print("✅ Додано:", file_name)
    else:
        print("⚠️ Не знайдено:", file_name)

shutil.make_archive(
    "Task_001_Бабенко_EnergyAgent",
    "zip",
    "EnergyAgent_project"
)

print("\n✅ ZIP створено: Task_001_Бабенко_EnergyAgent.zip")

⚠️ Не знайдено: Task_001_Бабенко_EnergyAgent.ipynb
✅ Додано: energy_agent_core.py
✅ Додано: test_energy_agent.py
✅ Додано: trajectory.json
✅ Додано: energy_agent_checkpoints.sqlite
✅ Додано: README.md

✅ ZIP створено: Task_001_Бабенко_EnergyAgent.zip


In [51]:
from google.colab import files

files.download("Task_001_Бабенко_EnergyAgent.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>